# 01: Data Quality Check & Exploratory Data Analysis (EDA)
**Owner:** Poojan (Person 1 - `poojan_p1`)
**Tasks:** P1.1 - P1.9

### Objective:
Thoroughly inspect the resume dataset, verify data quality (missing values, duplicates, text length), visualize the 24 classes, compute n-grams, and generate WordClouds.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
from wordcloud import WordCloud
from src.data_loader import load_resume_csv

# Load dataset
df = load_resume_csv()
print(f'Dataset shape: {df.shape}')
df.head()

## P1.1: Dataset Inspection
- 2,484 records, 4 columns (ID, Resume_str, Resume_html, Category)
- 24 unique classes

In [ ]:
print(f'Columns: {list(df.columns)}')
print(f'Shape: {df.shape}')
print(f'Categories: {df["Category"].nunique()}')
print(f'\nCategory list: {sorted(df["Category"].unique())}')
df.info()

## P1.2 & P1.3: Data Quality & Summary Statistics
- Check missing values and duplicate rows
- Calculate text length (character count and word count)

In [ ]:
print('=== Missing Values ===')
print(df.isnull().sum())

print('\n=== Empty Resume_str ===')
empty_mask = df['Resume_str'].isna() | (df['Resume_str'].astype(str).str.strip() == '')
print(f'Empty Resume_str: {empty_mask.sum()}')

print('\n=== Exact Duplicates ===')
print(f'Duplicate rows (by Resume_str): {df.duplicated(subset=["Resume_str"]).sum()}')

print('\n=== Short Resumes (< 10 words) ===')
df['word_count'] = df['Resume_str'].astype(str).apply(lambda x: len(x.split()))
df['char_count'] = df['Resume_str'].astype(str).apply(len)
short_mask = df['word_count'] < 10
print(f'Records with < 10 words: {short_mask.sum()}')

print('\n=== Summary Statistics ===')
df[['char_count', 'word_count']].describe()

## P1.4: Class Distribution Across the 24 Categories

In [ ]:
class_counts = df['Category'].value_counts()
print(f'Most frequent:  {class_counts.index[0]} ({class_counts.iloc[0]})')
print(f'Least frequent: {class_counts.index[-1]} ({class_counts.iloc[-1]})')
print(f'Imbalance ratio: {class_counts.iloc[0] / class_counts.iloc[-1]:.2f}x')

plt.figure(figsize=(14, 8))
colors = sns.color_palette('viridis', len(class_counts))
bars = plt.barh(class_counts.index[::-1], class_counts.values[::-1], color=colors)
plt.xlabel('Number of Resumes')
plt.title('Resume Count by Category (24 Classes)', fontweight='bold')
for bar, val in zip(bars, class_counts.values[::-1]):
    plt.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, str(val), va='center')
plt.tight_layout()
plt.savefig('../reports/figures/p1_4_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## P1.5: Resume Length Distributions (Word Count & Character Count)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(df['word_count'], bins=50, kde=True, color='teal', ax=axes[0])
axes[0].set_title('Resume Word Count Distribution', fontweight='bold')
axes[0].set_xlabel('Word Count')
axes[0].axvline(df['word_count'].median(), color='red', linestyle='--', label=f'Median: {df["word_count"].median():.0f}')
axes[0].legend()

sns.histplot(df['char_count'], bins=50, kde=True, color='coral', ax=axes[1])
axes[1].set_title('Resume Character Count Distribution', fontweight='bold')
axes[1].set_xlabel('Character Count')
axes[1].axvline(df['char_count'].median(), color='red', linestyle='--', label=f'Median: {df["char_count"].median():.0f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/p1_5_length_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

# Box plot by category
plt.figure(figsize=(14, 7))
order = df.groupby('Category')['word_count'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='Category', y='word_count', order=order, palette='viridis')
plt.xticks(rotation=45, ha='right')
plt.title('Word Count Distribution per Category', fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/p1_5_wordcount_by_category.png', dpi=150, bbox_inches='tight')
plt.show()

## P1.6: Word Frequency Analysis (Top Unigrams)

In [ ]:
stop_words = set(stopwords.words('english'))

def simple_tokenize(text):
    if not isinstance(text, str): return []
    return [w.lower() for w in text.split() if w.isalpha() and len(w) >= 2]

all_tokens = []
for text in df['Resume_str']:
    all_tokens.extend(simple_tokenize(text))

filtered_tokens = [t for t in all_tokens if t not in stop_words]
filtered_freq = Counter(filtered_tokens)
top_30 = filtered_freq.most_common(30)

fig, ax = plt.subplots(figsize=(14, 6))
words, counts = zip(*top_30)
ax.barh(list(reversed(words)), list(reversed(counts)), color=sns.color_palette('crest', 30))
ax.set_xlabel('Frequency')
ax.set_title('Top 30 Most Frequent Words (After Stopword Removal)', fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/p1_6_top30_words_no_stopwords.png', dpi=150, bbox_inches='tight')
plt.show()

## P1.7: WordClouds (Overall + Per-Category)

In [ ]:
# Overall WordCloud
all_text = ' '.join(df['Resume_str'].astype(str).tolist())
wc = WordCloud(width=1200, height=600, max_words=200, background_color='white',
               colormap='viridis', stopwords=stop_words, random_state=42)
wc.generate(all_text)

plt.figure(figsize=(14, 7))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('WordCloud - All Resumes', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/p1_7_wordcloud_overall.png', dpi=150, bbox_inches='tight')
plt.show()

# Top 6 categories
top_6_cats = df['Category'].value_counts().index[:6]
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
for idx, cat in enumerate(top_6_cats):
    ax = axes.flatten()[idx]
    cat_text = ' '.join(df[df['Category']==cat]['Resume_str'].astype(str).tolist())
    wc_cat = WordCloud(width=600, height=400, max_words=100, background_color='white',
                       colormap='plasma', stopwords=stop_words, random_state=42)
    wc_cat.generate(cat_text)
    ax.imshow(wc_cat, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(cat, fontweight='bold')
plt.suptitle('WordClouds - Top 6 Categories', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/p1_7_wordclouds_top6_categories.png', dpi=150, bbox_inches='tight')
plt.show()

## P1.8: N-Gram Analysis (Bigrams & Trigrams)

In [ ]:
def get_ngrams(tokens, n):
    return [' '.join(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

bigram_counter = Counter()
trigram_counter = Counter()
for text in df['Resume_str']:
    tokens = [t for t in simple_tokenize(text) if t not in stop_words]
    bigram_counter.update(get_ngrams(tokens, 2))
    trigram_counter.update(get_ngrams(tokens, 3))

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

top_20_bi = bigram_counter.most_common(20)
bi_ngrams, bi_counts = zip(*top_20_bi)
axes[0].barh(list(reversed(bi_ngrams)), list(reversed(bi_counts)), color=sns.color_palette('rocket', 20))
axes[0].set_xlabel('Frequency')
axes[0].set_title('Top 20 Bigrams', fontweight='bold')

top_20_tri = trigram_counter.most_common(20)
tri_ngrams, tri_counts = zip(*top_20_tri)
axes[1].barh(list(reversed(tri_ngrams)), list(reversed(tri_counts)), color=sns.color_palette('mako', 20))
axes[1].set_xlabel('Frequency')
axes[1].set_title('Top 20 Trigrams', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/p1_8_ngrams_bigrams_trigrams.png', dpi=150, bbox_inches='tight')
plt.show()

## P1.9: Class-Wise Vocabulary Analysis

In [ ]:
# Unique vocabulary per class
class_vocab = {}
for cat in sorted(df['Category'].unique()):
    cat_tokens = []
    for text in df[df['Category']==cat]['Resume_str']:
        cat_tokens.extend([t for t in simple_tokenize(text) if t not in stop_words])
    class_vocab[cat] = set(cat_tokens)

vocab_sizes = {cat: len(vocab) for cat, vocab in class_vocab.items()}
sorted_cats = sorted(vocab_sizes, key=vocab_sizes.get, reverse=True)

plt.figure(figsize=(14, 7))
plt.barh(list(reversed(sorted_cats)), [vocab_sizes[c] for c in reversed(sorted_cats)],
         color=sns.color_palette('flare', len(sorted_cats)))
plt.xlabel('Unique Vocabulary Size')
plt.title('Unique Vocabulary Size per Category (After Stopword Removal)', fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/p1_9_vocab_size_by_category.png', dpi=150, bbox_inches='tight')
plt.show()

# Mean word count by category
cat_stats = df.groupby('Category').agg(mean_words=('word_count','mean'), std_words=('word_count','std')).sort_values('mean_words', ascending=False)
plt.figure(figsize=(12, 6))
plt.bar(range(len(cat_stats)), cat_stats['mean_words'], yerr=cat_stats['std_words'], capsize=3,
        color=sns.color_palette('coolwarm', len(cat_stats)), alpha=0.85)
plt.xticks(range(len(cat_stats)), cat_stats.index, rotation=45, ha='right')
plt.ylabel('Mean Word Count (+/- 1 SD)')
plt.title('Mean Resume Length by Category', fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/p1_9_mean_wordcount_by_category.png', dpi=150, bbox_inches='tight')
plt.show()